In [ ]:
"""
Visualize ILI Component Detection dataset and training pipeline.

Shows: (1) raw + augmented stacked per sample, (2) class distribution.
"""
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer

from utils import load_config
from Dataset import register_detection_datasets

In [ ]:
config = load_config('config.yaml')

# Register datasets (uses Data/data.json + Data/images/, stratified train/val/test split)
n_total, n_train, n_val, n_test, num_classes, class_names = register_detection_datasets(config)
print(f"Datasets: total={n_total}, train={n_train}, val={n_val}, test={n_test}")
print(f"Classes ({num_classes}): {class_names}")

dataset_dicts = DatasetCatalog.get("data_detection_train")
metadata = MetadataCatalog.get("data_detection_train")

In [ ]:
# === Single visualization cell: class distribution + original vs augmented ===
from collections import Counter

num_samples = 6
samples = random.sample(dataset_dicts, min(num_samples, len(dataset_dicts)))
cat_counts = Counter()

for d in dataset_dicts:
    for ann in d.get("annotations", []):
        cid = ann["category_id"]
        cat_counts[metadata.thing_classes[cid]] += 1
print("Instances per class:")
for name, count in cat_counts.most_common():
    print(f"  {name}: {count}")

fig1, ax_bar = plt.subplots(figsize=(10, 4))
names, counts = zip(*cat_counts.most_common()) if cat_counts else ([], [])
bars = ax_bar.bar(names, counts, color=plt.cm.Set3(np.linspace(0, 1, max(1, len(names)))))
ax_bar.set_ylabel("Count")
ax_bar.set_title("Class distribution")
ax_bar.tick_params(axis="x", rotation=45)
for bar, c in zip(bars, counts):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, str(c), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ILI-only augmentation for visualization (avoids Detectron2 transforms that alter colors)
from Dataset.augmentations import apply_track_shift, apply_circular_roll

num_tracks = config.get("num_tracks", 22)
max_track_shift = config.get("max_track_shift", 15)
split_wrapped = config.get("split_wrapped_boxes", True)

n_cols = min(3, num_samples)
n_rows = int(np.ceil(num_samples / n_cols))
fig2, axes = plt.subplots(n_rows * 2, n_cols, figsize=(5 * n_cols, 5 * n_rows * 2))
axes = np.array(axes).reshape(n_rows * 2, n_cols)
for i, d in enumerate(samples):
    col, row_base = i % n_cols, 2 * (i // n_cols)
    ax_orig, ax_aug = axes[row_base, col], axes[row_base + 1, col]
    # Original
    img_orig = cv2.imread(d["file_name"])[:, :, ::-1]
    v_orig = Visualizer(img_orig, metadata=metadata, scale=0.8)
    ax_orig.imshow(v_orig.draw_dataset_dict(d).get_image())
    ax_orig.set_title("Original" if col == 0 else "")
    ax_orig.axis("off")
    # Augmented: ILI-only (track shift + circular roll) to preserve grayscale colors
    try:
        img_aug = cv2.imread(d["file_name"])
        h, w = img_aug.shape[:2]
        annos = [a for a in d.get("annotations", []) if a.get("iscrowd", 0) == 0]
        if annos:
            img_aug, annos = apply_track_shift(img_aug, annos, num_tracks, max_track_shift, h, w)
            img_aug, annos = apply_circular_roll(img_aug, annos, num_tracks, h, w, split_wrapped=split_wrapped)
        d_aug_viz = {"height": h, "width": w, "annotations": annos}
        img_aug_rgb = img_aug[:, :, ::-1]
        v_aug = Visualizer(img_aug_rgb, metadata=metadata, scale=0.8)
        ax_aug.imshow(v_aug.draw_dataset_dict(d_aug_viz).get_image())
        ax_aug.set_title("Augmented" if col == 0 else "")
    except Exception as e:
        ax_aug.imshow(img_orig)
        ax_aug.set_title(str(e)[:30] if col != 0 else "Augmented (error)", fontsize=8)
    ax_aug.axis("off")
for j in range(num_samples, n_cols * n_rows):
    r, c = 2 * (j // n_cols), j % n_cols
    axes[r, c].axis("off")
    axes[r + 1, c].axis("off")
plt.suptitle("Original (top) vs Augmented (bottom)", fontsize=14)
plt.tight_layout()
plt.show()